In [2]:
from netgen.occ import *
from ngsolve import *
from ngsolve.solvers import *
from ngsolve.webgui import Draw

## 3D Pendulum

In [3]:
# Define Parameters
length = 0.35
r_rod = 0.025
r_head = 0.075

# Define Points
base_rod_pt = (0, 0, 0)
top_rod_pt = (0, length, 0)


rod = Cylinder(base_rod_pt, Y, r_rod, length)
rod.faces.Min(Y).name = "base"

head = Sphere(top_rod_pt, r_head)
vec = gp_Vec(0, length, 0)
head.Move(vec)

shape = rod + head

mesh = Mesh(OCCGeometry(shape, dim=3).GenerateMesh(maxh=2))
mesh.Curve(3)
Draw(mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [ ]:
mu = 0.5e6  # shear modulus
lam = 1e6  # lame parameter


def C_3D(u):
    F = Id(3) + Grad(u)
    return F.trans * F


def NeoHooke_3D(C):
    return 0.5 * mu * (Trace(C - Id(3)) + 2 * mu / lam * Det(C) ** (-lam / 2 / mu) - 1)


V = VectorH1(mesh, order=2)
Q = NumberSpace(mesh, definedon=mesh.Boundaries("base"))

fes = V * Q**3
(u, q), (v, p) = fes.TnT()

gfut = GridFunction(V, multidim=0)

# define the needed GridFunctiond required in the scheme
gfu = GridFunction(fes)
gfv = GridFunction(fes)
gfa = GridFunction(fes)

gfuold = GridFunction(fes)
gfvold = GridFunction(fes)
gfaold = GridFunction(fes)

bfa = BilinearForm(fes)
bfa += Variation(NeoHooke_3D(C_3D(u)) * dx).Compile()
bfa += (InnerProduct(u, p) + InnerProduct(v, q)) * ds("base")  # we add the constraints

tau = 0.025  # time step size
tend = 3
rho = 7e3
force = CF((0, -1, 0))  # gravity force

In [6]:
from math import cos, pi, sin

theta = -45.0 * pi / 180.0
c, s = cos(theta), sin(theta)

# rotation pivot (e.g., hole center or origin)
cx, cy, cz = 0.0, 0.0, 0.0

xr = x - cx
yr = y - cy
zr = z - cz

# displacement for pure rotation (no scaling)
u_rot = CF(((c - 1.0) * xr - s * yr, s * xr + (c - 1.0) * yr, 0.0))


gfu.components[0].Set(u_rot)
gfuold.vec[:] = gfu.vec

In [7]:
vel_new = 2 / tau * (u - gfuold.components[0]) - gfvold.components[0]
acc_new = 2 / tau * (vel_new - gfvold.components[0]) - gfaold.components[0]

# need to add to the bilinear form since it depends on the current valurs of the GridFunctions

bfa += acc_new * v * dx
bfa += -force * v * dx

In [8]:
gfut.AddMultiDimComponent(gfu.components[0].vec)
scene = Draw(gfu.components[0], mesh, "deformation", deformation=True)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [9]:
tau = 0.025  # time step size
tend = 0.75
rho = 7e3
force = CF((0, -1, 0))  # gravity force


vel_new = 2 / tau * (u - gfuold.components[0]) - gfvold.components[0]
acc_new = 2 / tau * (vel_new - gfvold.components[0]) - gfaold.components[0]

# need to add to the bilinear form since it depends on the current valurs of the GridFunctions

bfa += acc_new * v * dx
bfa += -force * v * dx

In [10]:
from ngsolve.solvers import Newton

gfut.AddMultiDimComponent(gfu.components[0].vec)
t = 0
i = 1
with TaskManager():
    while t < tend:
        i += 1
        t += tau
        Newton(a=bfa, u=gfu, printing=False, inverse="sparsecholesky")

        if i % 5 == 0:
            # scene.Redraw()
            gfut.AddMultiDimComponent(gfu.components[0].vec)
        gfv.vec[:] = 2 / tau * (gfu.vec - gfuold.vec) - gfvold.vec
        gfa.vec[:] = 2 / tau * (gfv.vec - gfvold.vec) - gfaold.vec

        gfuold.vec[:] = gfu.vec
        gfvold.vec[:] = gfv.vec
        gfaold.vec[:] = gfa.vec

KeyboardInterrupt: 

In [11]:
settings = {"Multidim": {"speed": 1}}

Draw(
    gfut,
    mesh,
    interpolate_multidim=True,
    deformation=True,
    animate=True,
    autoscale=False,
    min=0,
    max=2,
    settings=settings,
);

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Multidim': {'speed': 1}}, '…